# Phase 2.1 — Reasoning Attention Calibration Distillation

We use the **46 Golden Tasks** identified in Phase 2 to train a lightweight
attention adapter (`U_q`, `U_k`) on the frozen 0-shot student model so that
its internal attention maps approximate the 4-shot teacher's representations.

- **Training set**: 38 golden tasks (teacher_prompt → hidden state target)
- **Test set**: 8 held-out golden tasks (never seen during training)
- **Adapter**: `CalibratedAttention` injected at every Llama layer (~393K trainable params)
- **Loss**: MSE(hidden) + 0.5×KL(probs) + 0.001×L2(adapter weights)
- **Evaluation**: does the 0-shot student, after calibration, predict the right MC letter?

## Step 1 — Install Dependencies

In [1]:
!pip install transformers torch -q

## Step 2 — Setup: Load Models (conditional to avoid OOM)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def load_model():
    return AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        attn_implementation="eager",   # disable SDPA so CalibratedAttention is called
    ).to(device)

if 'teacher_model' not in globals():
    teacher_model = load_model()
    for p in teacher_model.parameters():
        p.requires_grad = False
    teacher_model.eval()
    print("Teacher model loaded and frozen.")
else:
    print("Teacher model already in memory — skipping reload.")

if 'student_model' not in globals():
    student_model = load_model()
    for p in student_model.parameters():
        p.requires_grad = False
    student_model.eval()
    print("Student model loaded and frozen.")
else:
    print("Student model already in memory — skipping reload.")

n_params = sum(p.numel() for p in teacher_model.parameters())
print(f"Parameters per model : {n_params:,}")
print("Both models ready.")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Teacher model loaded and frozen.


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

Student model loaded and frozen.
Parameters per model : 1,711,376,384
Both models ready.


## Step 3 — MC Token IDs and Predict Helper

In [3]:
# SmolLM2 tokenises single uppercase letters as single tokens
A_ID = tokenizer.encode("A", add_special_tokens=False)[0]
B_ID = tokenizer.encode("B", add_special_tokens=False)[0]
C_ID = tokenizer.encode("C", add_special_tokens=False)[0]
D_ID = tokenizer.encode("D", add_special_tokens=False)[0]
MC_IDS     = [A_ID, B_ID, C_ID, D_ID]
MC_LETTERS = ["A",  "B",  "C",  "D"]

print(f"Token IDs  →  A={A_ID}  B={B_ID}  C={C_ID}  D={D_ID}")


def format_prompt(text: str) -> str:
    """Apply SmolLM2 chat template and append 'Answer:' so the model
    predicts the MC letter as the very next token."""
    messages = [{"role": "user", "content": text}]
    return (
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        + "Answer:"
    )


def predict_mc(model, prompt_text: str):
    """
    Single forward pass.  Returns (predicted_letter, probability).
    Probability is taken from the softmax over the full vocabulary,
    restricted to the four MC token positions.
    """
    formatted = format_prompt(prompt_text)
    inputs    = tokenizer(formatted, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[:, -1, :].float()
        probs  = F.softmax(logits, dim=-1)
    mc_probs = [probs[0, tid].item() for tid in MC_IDS]
    best_idx = mc_probs.index(max(mc_probs))
    return MC_LETTERS[best_idx], mc_probs[best_idx]


print("predict_mc() defined.")

Token IDs  →  A=49  B=50  C=51  D=52
predict_mc() defined.


## Step 4 — Golden Tasks Data and Train/Test Split

38 tasks are used for training the adapter; 8 are held out for evaluation.
The split is fixed with `random.seed(42)` for reproducibility.

In [4]:
import json
import random

raw_golden_json = r"""[
  {
    "id": 2,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 5,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 6,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 14,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 15,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 21,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((9-3)*4)+2\nOptions:\nA) 22\nB) 24\nC) 25\nD) 26",
    "answer": "D"
  },
  {
    "id": 24,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 30,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (4,1)\nWest 2\nNorth 4\nSouth 2\nOptions:\nA) (0,3)\nB) (0,6)\nC) (2,3)\nD) (4,1)",
    "answer": "C"
  },
  {
    "id": 34,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 36,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,1)\nWest 2\nEast 4\nWest 5\nEast 2\nNorth 1\nOptions:\nA) (1,-1)\nB) (1,3)\nC) (4,2)\nD) (7,1)",
    "answer": "C"
  },
  {
    "id": 46,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 52,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n(100/10)+(3*4)\nOptions:\nA) 18\nB) 20\nC) 21\nD) 22",
    "answer": "D"
  },
  {
    "id": 55,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 63,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(((())))\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 65,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(True OR False) AND (True AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 74,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,2)\nNorth 2\nSouth 5\nSouth 4\nOptions:\nA) (3,-6)\nB) (5,-4)\nC) (5,-5)\nD) (6,-8)",
    "answer": "C"
  },
  {
    "id": 86,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 88,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,0)\nEast 5\nEast 3\nSouth 5\nOptions:\nA) (10,-3)\nB) (11,-5)\nC) (14,-3)\nD) (8,-3)",
    "answer": "B"
  },
  {
    "id": 92,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()(())()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 95,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,6)\nNorth 2\nEast 4\nSouth 1\nNorth 2\nOptions:\nA) (2,7)\nB) (4,12)\nC) (5,9)\nD) (8,9)",
    "answer": "C"
  },
  {
    "id": 98,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 106,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n((()()()))()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 116,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (6,4)\nNorth 3\nNorth 1\nSouth 5\nOptions:\nA) (4,2)\nB) (5,6)\nC) (6,3)\nD) (6,6)",
    "answer": "C"
  },
  {
    "id": 121,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,1)\nWest 4\nNorth 2\nNorth 5\nSouth 2\nWest 2\nOptions:\nA) (-3,5)\nB) (-3,6)\nC) (-5,9)\nD) (0,7)",
    "answer": "B"
  },
  {
    "id": 123,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,0)\nSouth 5\nSouth 5\nWest 3\nOptions:\nA) (0,-8)\nB) (1,-10)\nC) (2,-10)\nD) (5,-13)",
    "answer": "C"
  },
  {
    "id": 124,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 125,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 130,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 135,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False AND False) OR (False OR False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 138,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()(()())()()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 140,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\n(False OR NOT True) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 141,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after each swap.\n\nExample 1\nAlice has the red ball.\nBob has the blue ball.\nSwap Alice and Bob.\nWho has the red ball?\nOptions:\nA) Bob\nB) Emma\nC) Eva\nD) Ryan\nAnswer: A\n\nExample 2\nTom has the key.\nEmma has the coin.\nSwap Tom and Emma.\nWho has the coin?\nOptions:\nA) Alice\nB) Bob\nC) Emma\nD) Tom\nAnswer: D\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Charlie\nB) Eva\nC) Kate\nD) Mia",
    "student_prompt": "Task: Track the ownership of objects after each swap.\n\nQuestion\nEva has the camera.\nKate has the hat.\nSwap Eva and Kate.\nWho has the camera?\nOptions:\nA) Alice\nB) Eva\nC) Kate\nD) Mia",
    "answer": "C"
  },
  {
    "id": 145,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True OR True) AND False\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 146,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,3)\nEast 5\nEast 4\nNorth 5\nSouth 1\nEast 2\nOptions:\nA) (13,5)\nB) (14,9)\nC) (16,7)\nD) (18,10)",
    "answer": "C"
  },
  {
    "id": 153,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n()()(())()\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 155,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (True AND True) OR (False AND False)\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 164,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (0,4)\nNorth 1\nSouth 5\nSouth 2\nSouth 3\nOptions:\nA) (-3,-2)\nB) (-3,-6)\nC) (0,-5)\nD) (2,-8)",
    "answer": "C"
  },
  {
    "id": 169,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (3,4)\nWest 2\nNorth 5\nSouth 4\nNorth 1\nEast 4\nOptions:\nA) (2,5)\nB) (2,7)\nC) (5,6)\nD) (7,8)",
    "answer": "C"
  },
  {
    "id": 171,
    "task": "tracking_shuffled_objects",
    "teacher_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nExample 1\nAlice has the ball.\nBob has the key.\nCharlie has the book.\nSwap Alice and Bob.\nSwap Bob and Charlie.\nWho has the ball?\nOptions:\nA) Amy\nB) Carl\nC) Charlie\nD) Eva\nAnswer: C\n\nExample 2\nTom has the coin.\nSam has the ring.\nEva has the pen.\nSwap Tom and Eva.\nSwap Eva and Sam.\nWho has the coin?\nOptions:\nA) Bob\nB) Emma\nC) John\nD) Sam\nAnswer: D\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "student_prompt": "Task: Track the ownership of objects after multiple swaps.\n\nQuestion\nDavid has the hat.\nJohn has the doll.\nJack has the badge.\nSwap John and Jack.\nSwap David and John.\nWho has the hat?\nOptions:\nA) David\nB) Jack\nC) John\nD) Mike",
    "answer": "C"
  },
  {
    "id": 172,
    "task": "dyck_language",
    "teacher_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nExample 1\n(()())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: A\n\nExample 2\n((())\nOptions:\nA) Balanced\nB) Unbalanced\nAnswer: B\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "student_prompt": "Task: Determine whether the parentheses are Balanced or Unbalanced.\n\nQuestion\n(()())\nOptions:\nA) Balanced\nB) Unbalanced",
    "answer": "A"
  },
  {
    "id": 173,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  },
  {
    "id": 174,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n((20/4)+7)*3\nOptions:\nA) 32\nB) 34\nC) 35\nD) 36",
    "answer": "D"
  },
  {
    "id": 178,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (5,4)\nNorth 5\nEast 4\nEast 2\nSouth 5\nOptions:\nA) (10,5)\nB) (11,4)\nC) (8,1)\nD) (8,4)",
    "answer": "B"
  },
  {
    "id": 180,
    "task": "navigate",
    "teacher_prompt": "Task: Determine the final coordinates.\n\nExample 1\nStart (0,0)\nNorth 2\nEast 3\nOptions:\nA) (2,1)\nB) (3,2)\nC) (3,4)\nD) (5,5)\nAnswer: B\n\nExample 2\nStart (4,1)\nWest 2\nSouth 1\nOptions:\nA) (2,0)\nB) (2,1)\nC) (3,4)\nD) (5,5)\nAnswer: A\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "student_prompt": "Task: Determine the final coordinates.\n\nQuestion\nStart (1,1)\nWest 5\nNorth 5\nEast 1\nWest 1\nOptions:\nA) (-1,9)\nB) (-2,3)\nC) (-4,6)\nD) (-5,6)",
    "answer": "C"
  },
  {
    "id": 194,
    "task": "multistep_arithmetic",
    "teacher_prompt": "Task: Solve the arithmetic expression.\n\nExample 1\n(4+2)*3\nOptions:\nA) 17\nB) 18\nC) 19\nD) 20\nAnswer: B\n\nExample 2\n10-(3*2)\nOptions:\nA) 3\nB) 4\nC) 5\nD) 6\nAnswer: B\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "student_prompt": "Task: Solve the arithmetic expression.\n\nQuestion\n7+(18/3)-2\nOptions:\nA) 7\nB) 9\nC) 10\nD) 11",
    "answer": "D"
  },
  {
    "id": 199,
    "task": "boolean_expressions",
    "teacher_prompt": "Task: Evaluate the Boolean expression.\n\nExample 1\n(True AND False) OR True\nOptions:\nA) False\nB) True\nAnswer: B\n\nExample 2\n(False OR False) AND True\nOptions:\nA) False\nB) True\nAnswer: A\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "student_prompt": "Task: Evaluate the Boolean expression.\n\nQuestion\nNOT (False OR NOT False) AND True\nOptions:\nA) False\nB) True",
    "answer": "A"
  }
]"""
golden_tasks = json.loads(raw_golden_json)

# Deterministic 38/8 split
random.seed(42)
shuffled = golden_tasks.copy()
random.shuffle(shuffled)

TRAIN_TASKS = shuffled[:38]
TEST_TASKS  = shuffled[38:]

print(f"Golden tasks loaded : {len(golden_tasks)}")
print(f"Training tasks      : {len(TRAIN_TASKS)}")
print(f"Test tasks          : {len(TEST_TASKS)}")
print()
print("Test task IDs:")
for t in TEST_TASKS:
    print(f"  ID {t['id']:3d} | {t['task']:<30} | Answer: {t['answer']}")

Golden tasks loaded : 46
Training tasks      : 38
Test tasks          : 8

Test task IDs:
  ID  24 | dyck_language                  | Answer: A
  ID  34 | boolean_expressions            | Answer: A
  ID  65 | boolean_expressions            | Answer: A
  ID  74 | navigate                       | Answer: C
  ID  88 | navigate                       | Answer: B
  ID   5 | dyck_language                  | Answer: A
  ID  30 | navigate                       | Answer: C
  ID 173 | boolean_expressions            | Answer: A


## Step 5 — Baseline: Student Predictions BEFORE Adapter Injection

We record the pure 0-shot student predictions now, while the model is
completely unmodified (no `CalibratedAttention` yet).

In [5]:
print("Capturing baseline predictions (unmodified student, 0-shot) ...")
baseline_preds = {}

for t in TEST_TASKS:
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct = (pred == t["answer"])
    baseline_preds[t["id"]] = {"pred": pred, "prob": prob, "correct": correct}
    mark = "✅" if correct else "❌"
    print(f"  Task {t['id']:3d} | {t['task']:<30} | True: {t['answer']} | "
          f"Pred: {pred} ({prob:.2f}) {mark}")

n_correct = sum(1 for v in baseline_preds.values() if v["correct"])
print(f"\nBaseline accuracy: {n_correct}/{len(TEST_TASKS)} = {n_correct/len(TEST_TASKS):.0%}")

Capturing baseline predictions (unmodified student, 0-shot) ...
  Task  24 | dyck_language                  | True: A | Pred: B (0.00) ❌
  Task  34 | boolean_expressions            | True: A | Pred: B (0.00) ❌
  Task  65 | boolean_expressions            | True: A | Pred: B (0.00) ❌
  Task  74 | navigate                       | True: C | Pred: B (0.00) ❌
  Task  88 | navigate                       | True: B | Pred: C (0.00) ❌
  Task   5 | dyck_language                  | True: A | Pred: B (0.00) ❌
  Task  30 | navigate                       | True: C | Pred: B (0.00) ❌
  Task 173 | boolean_expressions            | True: A | Pred: B (0.00) ❌

Baseline accuracy: 0/8 = 0%


## Step 6 — RoPE Utilities

SmolLM2 uses Rotary Position Embeddings. We implement the rotation here
so we are not dependent on transformers internals across versions.

In [6]:
def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    """
    Apply rotary position embeddings to Q and K.
    cos/sin: (1, 1, T, head_dim) or (B, 1, T, head_dim)
    q, k:    (B, H, T, head_dim)
    """
    if cos.dim() == 3:          # (1, T, D) — older API
        cos = cos.unsqueeze(1)  # (1, 1, T, D)
        sin = sin.unsqueeze(1)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


print("RoPE utilities defined.")

RoPE utilities defined.


## Step 7 — CalibratedAttention Adapter + Injection

Injects two low-rank trainable matrices `U_q` and `U_k` (rank=4) at every
Llama attention layer.  The frozen model weights are never touched.

The adapter computes:
```
delta  = (Q_rot @ U_q) @ (K_rot @ U_k)^T
S'     = S_frozen + delta
output = softmax(S') @ V
```
Only `U_q` and `U_k` receive gradients.

In [7]:
class CalibratedAttention(nn.Module):
    """
    Wraps a frozen LlamaAttention and injects:
        S' = S + delta,   delta = (Q_rot @ U_q) @ (K_rot @ U_k)^T
    where Q_rot and K_rot are Q and K AFTER RoPE.
    Only U_q and U_k (float32) are trainable.
    o_proj is NOT wrapped in no_grad so gradients flow through to U_q/U_k.
    """
    def __init__(self, original_attn, rank: int = 4):
        super().__init__()
        self.original_attn    = original_attn
        self.config           = getattr(original_attn, "config", None)

        # Robustly fetch attributes that changed location across transformers versions
        self.num_heads        = getattr(original_attn, "num_heads", None) or self.config.num_attention_heads
        self.num_kv_heads     = getattr(original_attn, "num_key_value_heads", None) or self.config.num_key_value_heads
        self.hidden_size      = getattr(original_attn, "hidden_size", None) or self.config.hidden_size
        self.head_dim         = getattr(original_attn, "head_dim", None) or (self.hidden_size // self.num_heads)
        self.num_kv_groups    = self.num_heads // self.num_kv_heads

        self.expected_tuple_len = None

        # float32 trainable adapter weights — small and fast to optimise
        self.U_q = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        self.U_k = nn.Parameter(torch.zeros(self.num_heads, self.head_dim, rank))
        nn.init.normal_(self.U_q, std=0.02)
        nn.init.normal_(self.U_k, std=0.02)

    def forward(
        self,
        hidden_states,
        attention_mask=None,
        position_ids=None,
        past_key_value=None,
        output_attentions=False,
        use_cache=False,
        cache_position=None,
        position_embeddings=None,   # (cos, sin) passed by newer transformers versions
        **kwargs,
    ):
        # Detect expected return-tuple length exactly once (handles transformers API differences)
        if self.expected_tuple_len is None:
            with torch.no_grad():
                dummy_out = self.original_attn(
                    hidden_states=hidden_states,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_value=past_key_value,
                    output_attentions=output_attentions,
                    use_cache=use_cache,
                    cache_position=cache_position,
                    position_embeddings=position_embeddings,
                    **kwargs
                )
                self.expected_tuple_len = len(dummy_out) if isinstance(dummy_out, tuple) else 1

        B, T, _ = hidden_states.shape

        # ── 1. Frozen QKV projections ─────────────────────────────────────
        with torch.no_grad():
            Q = self.original_attn.q_proj(hidden_states)  # (B, T, H*D)
            K = self.original_attn.k_proj(hidden_states)  # (B, T, kv_H*D)
            V = self.original_attn.v_proj(hidden_states)  # (B, T, kv_H*D)

        # Reshape to (B, heads, T, head_dim)
        Q = Q.view(B, T, self.num_heads,    self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # ── 2. Apply RoPE to Q and K (frozen) ────────────────────────────
        with torch.no_grad():
            if position_embeddings is not None:
                cos, sin = position_embeddings
            else:
                cos, sin = self.original_attn.rotary_emb(V, position_ids)
            Q, K = apply_rotary_pos_emb(Q, K, cos.float(), sin.float())

            # GQA: expand K and V to match num_heads
            if self.num_kv_groups > 1:
                K = K.repeat_interleave(self.num_kv_groups, dim=1)
                V = V.repeat_interleave(self.num_kv_groups, dim=1)

        # Detach — only U_q and U_k carry gradients
        Q = Q.detach().float()   # (B, H, T, D) float32
        K = K.detach().float()
        V = V.detach().float()

        # ── 3. Delta — the ONLY differentiable path ───────────────────────
        A     = torch.einsum('bhtd,hdr->bhtr', Q, self.U_q)        # (B,H,T,rank)
        Bm    = torch.einsum('bhtd,hdr->bhtr', K, self.U_k)        # (B,H,T,rank)
        delta = torch.matmul(A, Bm.transpose(-1, -2))               # (B,H,T,T)

        # ── 4. Frozen baseline attention scores ───────────────────────────
        with torch.no_grad():
            S      = torch.matmul(Q, K.transpose(-1, -2)) / (self.head_dim ** 0.5)
            causal = torch.tril(torch.ones(T, T, device=device)).bool()
            S      = S.masked_fill(~causal, torch.finfo(S.dtype).min)

        # ── 5. Inject delta ───────────────────────────────────────────────
        S_prime = S + delta
        if attention_mask is not None:
            S_prime = S_prime + attention_mask.float()
        attn_weights = F.softmax(S_prime, dim=-1)                   # (B,H,T,T)

        # ── 6. Weighted sum and merge heads ──────────────────────────────
        attn_output = torch.matmul(attn_weights, V)                 # (B,H,T,D)
        attn_output = attn_output.transpose(1, 2).contiguous()      # (B,T,H,D)
        attn_output = attn_output.view(B, T, self.num_heads * self.head_dim)

        # Cast back to model dtype before o_proj
        attn_output = attn_output.to(dtype=hidden_states.dtype)

        # ── 7. Output projection (gradient flows through here to U_q/U_k) ─
        attn_output = self.original_attn.o_proj(attn_output)        # (B,T,hidden)

        return (attn_output, None, None, None)[:self.expected_tuple_len]


def inject_calibration(model, rank=4):
    """Replace every LlamaAttention with CalibratedAttention. Returns trainable params."""
    trainable_params = []
    for layer in model.model.layers:
        cal = CalibratedAttention(layer.self_attn, rank=rank).to(device)
        layer.self_attn = cal
        trainable_params.extend([cal.U_q, cal.U_k])
    return trainable_params


# Only inject if not already done (re-running this cell re-injects cleanly
# because student_model is reset by the conditional load above)
if 'trainable_params' not in globals():
    trainable_params = inject_calibration(student_model, rank=4)
    print("Injection complete.")
else:
    print("Adapter already injected — skipping to avoid double-wrapping.")
    print("If you want a fresh injection, restart the kernel and re-run from Step 2.")

n_trainable = sum(p.numel() for p in trainable_params)
print(f"Trainable scalars : {n_trainable:,}  (U_q + U_k across all layers)")

Injection complete.
Trainable scalars : 393,216  (U_q + U_k across all layers)


## Step 8 — Gradient Check

In [8]:
chk_task   = TRAIN_TASKS[0]
inputs_chk = tokenizer(format_prompt(chk_task["student_prompt"]),
                       return_tensors="pt").to(device)
out_chk    = student_model(**inputs_chk, output_hidden_states=True)
hs_chk     = out_chk.hidden_states[-1][:, -1, :]

assert hs_chk.requires_grad, "❌ Gradient check FAILED — U_q/U_k not in graph!"
print("✅ Gradient check passed — U_q and U_k are in the computation graph.")

✅ Gradient check passed — U_q and U_k are in the computation graph.


## Step 9 — Pre-compute Teacher Targets

Run the **teacher model** once per training task using the 2-shot `teacher_prompt`.
We cache the final hidden state and full probability distribution.
The teacher is **never run again** after this cell.

In [9]:
print(f"Pre-computing teacher targets for {len(TRAIN_TASKS)} training tasks ...")
train_pairs = []

for i, t in enumerate(TRAIN_TASKS):
    # Teacher gets the 2-shot prompt; student gets the 0-shot prompt
    teacher_inputs = tokenizer(format_prompt(t["teacher_prompt"]),
                               return_tensors="pt").to(device)
    student_inputs = tokenizer(format_prompt(t["student_prompt"]),
                               return_tensors="pt").to(device)

    with torch.no_grad():
        out_t  = teacher_model(**teacher_inputs, output_hidden_states=True)
        t_hs   = out_t.hidden_states[-1][:, -1, :].float().detach()   # (1, hidden)
        t_prob = F.softmax(out_t.logits[:, -1, :].float(), dim=-1).detach()  # (1, vocab)

    train_pairs.append((student_inputs, t_hs, t_prob, t["answer"]))

    if (i + 1) % 10 == 0 or i == 0:
        print(f"  [{i+1:02d}/{len(TRAIN_TASKS)}] Task {t['id']:3d} | {t['task']}")

print(f"\nDone. {len(train_pairs)} (student_inputs, teacher_hidden, teacher_probs) pairs cached.")

Pre-computing teacher targets for 38 training tasks ...
  [01/38] Task  92 | dyck_language
  [10/38] Task  14 | dyck_language
  [20/38] Task 140 | boolean_expressions
  [30/38] Task  86 | multistep_arithmetic

Done. 38 (student_inputs, teacher_hidden, teacher_probs) pairs cached.


## Step 10 — Test Results BEFORE Training

Adapter is injected but weights are still random (not yet trained).

In [10]:
student_model.eval()
print("\n" + "="*72)
print("TEST SET — BEFORE TRAINING  (adapter injected, U_q/U_k not yet trained)")
print("="*72)

for t in TEST_TASKS:
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct    = (pred == t["answer"])
    base       = baseline_preds[t["id"]]
    mark       = "✅" if correct else "❌"
    base_mark  = "✅" if base["correct"] else "❌"
    print(f"Task {t['id']:3d} | {t['task']:<28} | True: {t['answer']} | "
          f"Baseline: {base['pred']} {base_mark} | Pre-train: {pred} ({prob:.2f}) {mark}")


TEST SET — BEFORE TRAINING  (adapter injected, U_q/U_k not yet trained)
Task  24 | dyck_language                | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  34 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  65 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  74 | navigate                     | True: C | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  88 | navigate                     | True: B | Baseline: C ❌ | Pre-train: C (0.00) ❌
Task   5 | dyck_language                | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task  30 | navigate                     | True: C | Baseline: B ❌ | Pre-train: B (0.00) ❌
Task 173 | boolean_expressions          | True: A | Baseline: B ❌ | Pre-train: B (0.00) ❌


## Step 11 — Training

Full-batch gradient descent: accumulate gradients over **all 38 training tasks**,
then perform **one weight update per epoch**.

In [11]:
EPOCHS = 300
LR     = 1e-3

optimizer = optim.AdamW(trainable_params, lr=LR, weight_decay=0.0)

print(f"Training for {EPOCHS} epochs  |  lr={LR}  |  full-batch ({len(train_pairs)} tasks/epoch)")
print(f"Loss = MSE(hidden) + 0.5*KL(probs) + 0.001*L2(adapter)\n")

for epoch in range(1, EPOCHS + 1):
    student_model.train()
    optimizer.zero_grad()
    total_loss = 0.0

    for s_inputs, t_hs, t_probs, _ in train_pairs:
        out    = student_model(**s_inputs, output_hidden_states=True)
        s_hs   = out.hidden_states[-1][:, -1, :].float()          # (1, hidden)
        s_prob = F.softmax(out.logits[:, -1, :].float(), dim=-1)  # (1, vocab)

        mse  = F.mse_loss(s_hs, t_hs)
        kl   = F.kl_div(s_prob.log(), t_probs, reduction='batchmean')
        l2   = sum(p.pow(2).sum() for p in trainable_params)
        loss = mse + 0.5 * kl + 0.001 * l2

        loss.backward()            # accumulate grads — do NOT zero inside loop
        total_loss += loss.item()

    torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
    optimizer.step()               # single weight update for the whole epoch

    if epoch == 1 or epoch % 25 == 0:
        print(f"Epoch {epoch:03d} | Avg Loss: {total_loss / len(train_pairs):.4f}")

Training for 300 epochs  |  lr=0.001  |  full-batch (38 tasks/epoch)
Loss = MSE(hidden) + 0.5*KL(probs) + 0.001*L2(adapter)

Epoch 001 | Avg Loss: 0.5166
Epoch 025 | Avg Loss: 0.0855
Epoch 050 | Avg Loss: 0.0407
Epoch 075 | Avg Loss: 0.0353
Epoch 100 | Avg Loss: 0.0308


KeyboardInterrupt: 

## Step 12 — Test Results AFTER Training

We compare three states side by side:
- **Baseline**: pure 0-shot before any adapter injection
- **After**: 0-shot student with trained `U_q`/`U_k` (still no demonstrations in prompt)

Any ❌→✅ flip means the adapter successfully transferred the teacher's reasoning attention.

In [12]:
student_model.eval()

print("\n" + "="*80)
print("TEST SET — AFTER TRAINING")
print("="*80)
print(f"{'ID':<6} {'Type':<30} {'True':>5} {'Baseline':>10} {'After':>10}  {'Change'}")
print("="*80)

after_correct   = 0
flipped         = 0

for t in TEST_TASKS:
    base       = baseline_preds[t["id"]]
    pred, prob = predict_mc(student_model, t["student_prompt"])
    correct    = (pred == t["answer"])

    if correct:
        after_correct += 1

    b_mark = "✅" if base["correct"] else "❌"
    a_mark = "✅" if correct else "❌"

    if not base["correct"] and correct:
        change = "← RECALIBRATED ✨"
        flipped += 1
    elif base["correct"] and not correct:
        change = "← REGRESSED ⚠️"
    else:
        change = ""

    print(f"{t['id']:<6} {t['task']:<30} {t['answer']:>5} "
          f"  {base['pred']} {b_mark}        {pred} {a_mark}  {change}")

print("="*80)
before_n = sum(1 for v in baseline_preds.values() if v["correct"])
print(f"\nBaseline (pure 0-shot)    : {before_n}/{len(TEST_TASKS)} correct ({before_n/len(TEST_TASKS):.0%})")
print(f"After calibration (0-shot) : {after_correct}/{len(TEST_TASKS)} correct ({after_correct/len(TEST_TASKS):.0%})")
print(f"Net recalibrated tasks     : {flipped:+d}")


TEST SET — AFTER TRAINING
ID     Type                            True   Baseline      After  Change
24     dyck_language                      A   B ❌        A ✅  ← RECALIBRATED ✨
34     boolean_expressions                A   B ❌        B ❌  
65     boolean_expressions                A   B ❌        B ❌  
74     navigate                           C   B ❌        B ❌  
88     navigate                           B   C ❌        C ❌  
5      dyck_language                      A   B ❌        B ❌  
30     navigate                           C   B ❌        D ❌  
173    boolean_expressions                A   B ❌        B ❌  

Baseline (pure 0-shot)    : 0/8 correct (0%)
After calibration (0-shot) : 1/8 correct (12%)
Net recalibrated tasks     : +1
